# MMDLQA Colab Quickstart

Run cells from top to bottom. Recommended workflow: keep code in GitHub, keep data in Google Drive. Colab clones/pulls code into `/content/MMDLQA`, then reads `input/sample_questions.xlsx`, `input/text_cleaning_output/`, and `input/raw/` from Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull Project Code

Edit `REPO_URL` to your GitHub repository. Edit `DRIVE_DATA_DIR` to the Drive folder that contains `input/sample_questions.xlsx`, `input/text_cleaning_output/`, and `input/raw/`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/MMDLQA.git'
PROJECT_DIR = '/content/MMDLQA'
DRIVE_DATA_DIR = '/content/drive/MyDrive/MMDLQA_data'

if not Path(PROJECT_DIR, '.git').exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    !git -C $PROJECT_DIR pull

os.chdir(PROJECT_DIR)
os.environ['MMDLQA_RAW_DIR'] = f'{DRIVE_DATA_DIR}/input/raw'
os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'] = f'{DRIVE_DATA_DIR}/input/text_cleaning_output'
os.environ['MMDLQA_USE_TEXT_CLEANING_OUTPUT'] = '1'
os.environ['MMDLQA_INCLUDE_RAW_FALLBACK'] = '1'
os.environ['MMDLQA_QUESTIONS'] = f'{DRIVE_DATA_DIR}/input/sample_questions.xlsx'
os.environ['MMDLQA_OUTPUT_DIR'] = f'{DRIVE_DATA_DIR}/output'
os.environ['MMDLQA_CACHE_DIR'] = f'{DRIVE_DATA_DIR}/output/cache'
os.environ['MMDLQA_SUBMISSION'] = f'{DRIVE_DATA_DIR}/output/submission.csv'

!pwd
!git status --short
print('raw:', os.environ['MMDLQA_RAW_DIR'])
print('cleaned:', os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'])
print('questions:', os.environ['MMDLQA_QUESTIONS'])
print('output:', os.environ['MMDLQA_OUTPUT_DIR'])

## 3. Install Dependencies

This can take a few minutes, especially Whisper.

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-vie tesseract-ocr-chi-sim libreoffice poppler-utils
!pip install -r requirements.txt

## 4. Check Data Layout

In [ ]:
!find "$MMDLQA_RAW_DIR" -type f | head -30
!python - <<'PY'
import os
from pathlib import Path
from collections import Counter
raw = Path(os.environ['MMDLQA_RAW_DIR'])
cleaned = Path(os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'])
questions = Path(os.environ['MMDLQA_QUESTIONS'])
files = list(raw.rglob('*'))
files = [p for p in files if p.is_file()]
manifest = cleaned / 'text_cleaning_manifest.csv'
print('files:', len(files))
print(Counter(p.suffix.lower() for p in files).most_common())
print('cleaned exists:', cleaned.exists(), cleaned)
print('cleaned by_file dirs:', len(list((cleaned / 'by_file').glob('*'))) if (cleaned / 'by_file').exists() else 0)
print('cleaned manifest exists:', manifest.exists(), manifest)
print('questions exists:', questions.exists(), questions)

## 5. Set OpenRouter Key

Use Colab Secrets if possible: left sidebar -> key icon -> add `OPENROUTER_API_KEY`. If not, paste it below for a quick test.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY') or ''
except Exception:
    pass

# Uncomment this line only for a quick local test if you do not use Colab Secrets.
# os.environ['OPENROUTER_API_KEY'] = 'PASTE_YOUR_KEY_HERE'

os.environ['MMDLQA_USE_MODEL_ROUTER'] = '1'
os.environ['MMDLQA_PRINT_QUESTION_METRICS'] = '1'
os.environ['MMDLQA_USE_EVIDENCE_SCANNER'] = '1'
os.environ['MMDLQA_EVIDENCE_SCAN_MAX_FILES'] = '24'
os.environ['MMDLQA_EVIDENCE_SCAN_IRRELEVANT_PATIENCE'] = '5'
os.environ['MMDLQA_RERANK_CANDIDATE_K'] = '36'
os.environ['MMDLQA_RERANK_TOP_K'] = '12'
os.environ['MMDLQA_AGENTIC_MAX_ROUNDS'] = '4'
os.environ['MMDLQA_MAX_QUESTION_COST_USD'] = '0.1'
os.environ['MMDLQA_PLANNER_MODEL'] = 'openai/gpt-5.6-terra'
os.environ['MMDLQA_RERANK_MODEL'] = 'openai/gpt-5.6-luna'
os.environ['MMDLQA_EXACT_MODEL'] = 'anthropic/claude-sonnet-5'
os.environ['MMDLQA_SYNTHESIS_MODEL'] = 'openai/gpt-5.6-sol-pro'
os.environ['MMDLQA_CRITIC_MODEL'] = 'anthropic/claude-sonnet-5'
os.environ['MMDLQA_CODER_MODEL'] = 'x-ai/grok-4.5'
os.environ['MMDLQA_VISION_MODEL'] = 'anthropic/claude-sonnet-5'
os.environ['MMDLQA_SCAN_TEXT_MODEL'] = 'openai/gpt-5.6-luna'
os.environ['MMDLQA_SCAN_TABLE_MODEL'] = 'x-ai/grok-4.5'
os.environ['MMDLQA_SCAN_DOCUMENT_MODEL'] = 'anthropic/claude-sonnet-5'
os.environ['MMDLQA_SCAN_IMAGE_MODEL'] = 'anthropic/claude-sonnet-5'
os.environ['MMDLQA_SCAN_AUDIO_MODEL'] = 'openai/gpt-5.6-luna'
os.environ['MMDLQA_SCAN_VIDEO_MODEL'] = 'openai/gpt-5.6-luna'
print('key set:', bool(os.environ.get('OPENROUTER_API_KEY')))
print('planner model:', os.environ['MMDLQA_PLANNER_MODEL'])
print('synthesis model:', os.environ['MMDLQA_SYNTHESIS_MODEL'])

## 6. Cheap Baseline Dry Run

No OpenRouter calls. This verifies ingestion, indexing, retrieval, and CSV output with the baseline runner.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
!python script/run_pipeline.py --rebuild-index --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 7. Agentic Dry Run

No OpenRouter calls. This verifies the multi-agent workflow, sentence-level RAG boundary, deterministic tools, static critic, and diagnostics.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
%env MMDLQA_USE_AGENTIC_MOE=0
%env MMDLQA_USE_AGENTIC_CRITIC=0
!python script/run_agentic.py --rebuild-index --limit 5 --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 8. Full Agentic Run With LLM

This uses OpenRouter for planner/MoE/critic calls. Keep `MMDLQA_USE_LLM_SUMMARIES=0` first to avoid captioning every image during indexing.

In [ ]:
%env MMDLQA_USE_LLM=1
%env MMDLQA_USE_LLM_RERANK=1
%env MMDLQA_USE_VISION_LLM=1
%env MMDLQA_USE_LLM_SUMMARIES=0
%env MMDLQA_USE_WHISPER=1
%env MMDLQA_USE_AGENTIC_PLANNER=1
%env MMDLQA_USE_AGENTIC_MOE=1
%env MMDLQA_USE_AGENTIC_CRITIC=1
%env MMDLQA_USE_AGENTIC_TOOLS=1
%env MMDLQA_USE_AGENTIC_CODER=1
%env MMDLQA_USE_EVIDENCE_SCANNER=1
%env MMDLQA_USE_CODER_PLANNER=0
%env MMDLQA_USE_MODEL_ROUTER=1
%env MMDLQA_PRINT_QUESTION_METRICS=1
%env MMDLQA_PLANNER_MODEL=openai/gpt-5.6-terra
%env MMDLQA_RERANK_MODEL=openai/gpt-5.6-luna
%env MMDLQA_EXACT_MODEL=anthropic/claude-sonnet-5
%env MMDLQA_SYNTHESIS_MODEL=openai/gpt-5.6-sol-pro
%env MMDLQA_CRITIC_MODEL=anthropic/claude-sonnet-5
%env MMDLQA_CODER_MODEL=x-ai/grok-4.5
%env MMDLQA_VISION_MODEL=anthropic/claude-sonnet-5
%env MMDLQA_SCAN_TEXT_MODEL=openai/gpt-5.6-luna
%env MMDLQA_SCAN_TABLE_MODEL=x-ai/grok-4.5
%env MMDLQA_SCAN_DOCUMENT_MODEL=anthropic/claude-sonnet-5
%env MMDLQA_SCAN_IMAGE_MODEL=anthropic/claude-sonnet-5
%env MMDLQA_SCAN_AUDIO_MODEL=openai/gpt-5.6-luna
%env MMDLQA_SCAN_VIDEO_MODEL=openai/gpt-5.6-luna
%env MMDLQA_AGENTIC_MAX_STEPS=5
%env MMDLQA_AGENTIC_MAX_ROUNDS=4
%env MMDLQA_EVIDENCE_SCAN_MAX_FILES=24
%env MMDLQA_EVIDENCE_SCAN_CHUNKS_PER_FILE=3
%env MMDLQA_EVIDENCE_SCAN_BATCH_SIZE=8
%env MMDLQA_EVIDENCE_SCAN_IRRELEVANT_PATIENCE=5
%env MMDLQA_RERANK_CANDIDATE_K=36
%env MMDLQA_RERANK_TOP_K=12
%env MMDLQA_MAX_QUESTION_SECONDS=0
%env MMDLQA_MAX_QUESTION_LLM_CALLS=0
%env MMDLQA_MAX_QUESTION_COST_USD=0.1
%env MMDLQA_MAX_QUESTION_RAG_QUERIES=0
%env MMDLQA_LLM_INPUT_COST_PER_MILLION_TOKENS=0
%env MMDLQA_LLM_OUTPUT_COST_PER_MILLION_TOKENS=0
!python -c "import os; assert os.environ.get('MMDLQA_USE_LLM') == '1', 'MMDLQA_USE_LLM must be 1 for full run'; assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY is missing'; print('LLM preflight ok')"
!python script/run_agentic.py --rebuild-index --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 9. Inspect Cost And Time

The run cell prints live per-question progress when `MMDLQA_PRINT_QUESTION_METRICS=1`. This cell renders the final metrics table from `diagnostics.jsonl`.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

out = Path(os.environ['MMDLQA_OUTPUT_DIR'])
summary_path = out / 'run_summary.json'
diagnostics_path = out / 'diagnostics.jsonl'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('mode:', summary.get('mode'))
print('questions:', summary.get('questions'))
print('indexed:', summary.get('indexed_files'), 'files /', summary.get('indexed_chunks'), 'chunks')
print('total elapsed sec:', summary.get('metrics', {}).get('total_elapsed_sec'))
print('total llm calls:', summary.get('metrics', {}).get('total_llm_calls'))
print('total failed llm calls:', summary.get('metrics', {}).get('total_failed_llm_calls'))
print('total estimated cost usd:', summary.get('metrics', {}).get('total_estimated_cost_usd'))

rows = []
for line in diagnostics_path.read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    answer = row.get('answer', {})
    metrics = answer.get('diagnostics', {}).get('metrics', {})
    agentic = answer.get('diagnostics', {}).get('agentic', {})
    scans = agentic.get('diagnostics', {}).get('evidence_scans', [])
    rows.append({
        'qid': answer.get('qid'),
        'elapsed_sec': metrics.get('elapsed_sec'),
        'llm_calls': metrics.get('llm_call_count'),
        'failed_llm': metrics.get('failed_llm_call_count'),
        'tokens': metrics.get('total_tokens'),
        'cost_usd': metrics.get('total_estimated_cost_usd'),
        'first_llm_error': str(metrics.get('first_llm_error', ''))[:160],
        'scan_batches': len(scans),
        'scanned_files': sum(s.get('processed_files', 0) for s in scans if isinstance(s, dict)),
        'answer': str(answer.get('answer', ''))[:120],
    })

df = pd.DataFrame(rows)
display(df)
display(df[['elapsed_sec', 'llm_calls', 'failed_llm', 'tokens', 'cost_usd', 'scanned_files']].describe())

## 10. Download Submission

In [ ]:
from google.colab import files
import os
files.download(os.environ['MMDLQA_SUBMISSION'])